# 05 — Gate 1: S0 construction (frequency-ensemble spec v0.9.1 §3.1, §7)

**Zero solves.** This notebook (1) runs the **v0.9.1 pre-freeze diagnostic** — the biomass
θ-tail capture rate from the archived a1/a0 solutions — which decides the carbon
within-block split; (2) derives the **S0–S4 (w, t) pairs** by block-level influence
budgeting in two-regime swing currency (D1, binding); and (3) freezes the derivation to
`spec/scenarios_v1.json`, the input the Gate-3 manifest consumes.

**Why S0 is not "a1 with a biomass tweak":** a1's w=1 vector was itself an implicit
influence profile — biomass held ~29% of discretionary swing (0.801 of ~2.73, the largest
single share), which is exactly the mechanism of the +8.4-pt leak. w=t is a **retired**
Gate-0 convention (spec §2.7); S0 re-derives *every* continuous weight so the four PROACT
blocks hold equal discretionary shares.

**Run order:** 02's closing section (climate realizations) first if not yet run; then this
notebook top-to-bottom. It pauses at the §A **checkpoint** for Ethan to confirm the split
before §B derives the scenarios. Kernel: `y2y-geo`.

In [1]:
# ---- bootstrap: find the repo root (this notebook lives below it) --------------------------
import importlib, json, pathlib, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook -- run from within the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec
for _m in (config, lc, ec):
    importlib.reload(_m)

SPEC = ROOT / "analyses" / "y2y" / "spec"
AUDIT_OBJ = ROOT / "analyses" / "y2y" / "audit" / "audit_objects"
CONSTS = json.loads((AUDIT_OBJ / "audit_constants.json").read_text())
T2 = pd.read_csv(AUDIT_OBJ / "feature_characterization.csv").set_index("feature")
assert CONSTS["constants"] == config.AUDIT, "config.AUDIT drifted from the frozen audit -- STOP"
print(f"frozen audit: {CONSTS['created_utc']}  n_pu = {CONSTS['n_pu']:,}")

frozen audit: 2026-08-26T22:32:08.502910+00:00  n_pu = 1,272,914


## §A — The v0.9.1 diagnostic: biomass θ-tail capture (zero solves)

The one open Gate-1 input. **Design intent to protect:** dense biomass *is* wanted — but a
weight already buys each feature's tail first (within-feature preference is
density-descending at any positive w; weight sets *depth*, not *order*), and m_soc's locked
claim co-captures overlapping biomass hotspots. So the question is empirical: **does the
biomass θ-tail (≥5× regional mean) escape the a1 selection?** — a1 being the S0-relevant
regime (m_soc targeted, biomass weight-levered).

**Pre-registered decision rule (frozen here BEFORE the numbers are seen; the rule reads the
a1 *mass-weighted* capture, since the design intent is dense mass):**

| a1 mass-weighted tail capture | verdict | within-block split |
|---|---|---|
| ≥ 0.90 | tail captured — concern discharged with evidence | **(a) mass-proportional** (regional SOC/biomass mass split) |
| 0.50 – 0.90 | tail partly escaping | **(b) equal (50/50)**, documented as evidence-driven |
| < 0.50 | tail escaping | **STOP** — the escalation (combined total-carbon feature) is a spec amendment decided with the chat, not a notebook branch |

The decomposition into **SOC-claim co-capture** (tail cells inside the m_soc θ-tail) vs
**independent selection** says *how* the tail gets bought — it is reported either way and
becomes the standing T1/E7 diagnostic (spec v0.9.1).

In [2]:
# ---- theta-tails on the PU grid, checked against the frozen T2 -----------------------------
HD = config.HANDOFF_DIR
pu = lc.pu_mask()
assert int(pu.sum()) == CONSTS["n_pu"], "PU count drifted from the frozen audit -- STOP"
theta = config.AUDIT["theta"]

bio = lc._read(HD / "irrecoverable_carbon_biomass.tif")
soc = lc._read(HD / "irrecoverable_carbon_m_soc.tif")
tails = {}
for name, arr in (("irrecoverable_carbon_biomass", bio), ("irrecoverable_carbon_m_soc", soc)):
    cut = theta * float(np.nanmean(arr[pu]))
    tails[name] = pu & (np.nan_to_num(arr, nan=-1.0) >= cut)
    t_area = tails[name].sum() / pu.sum()
    frozen = float(T2.loc[name, "theta_area"])
    print(f"{name}: cutoff {cut:.1f} t/ha | tail {100*t_area:.2f}% of PU (frozen T2 {100*frozen:.2f}%)")
    assert abs(t_area - frozen) < 2e-3, f"{name}: tail area drifted from the frozen T2 -- STOP"
bio_tail, soc_tail = tails["irrecoverable_carbon_biomass"], tails["irrecoverable_carbon_m_soc"]

irrecoverable_carbon_biomass: cutoff 105.6 t/ha | tail 1.17% of PU (frozen T2 1.17%)
irrecoverable_carbon_m_soc: cutoff 303.7 t/ha | tail 4.06% of PU (frozen T2 4.06%)


In [3]:
# ---- the diagnostic: tail capture in a1 (S0-relevant regime) + a0 (reference) --------------
def sel(run):
    a = ec._alloc(config.RESULTS_DIR / run / "portfolio.tif")
    return np.nan_to_num(a) > 0.5

S = {"a0_control": sel("iter8_y2y_a0_control"), "a1_protocol": sel("iter8_y2y_a1_protocol")}
mass_tot = float(np.nansum(bio[bio_tail]))
rows = []
for arm, s in S.items():
    hit = bio_tail & s
    rows.append(dict(
        arm=arm,
        tail_cells_selected=f"{int(hit.sum()):,}/{int(bio_tail.sum()):,}",
        capture_cells=hit.sum() / bio_tail.sum(),
        capture_mass=float(np.nansum(bio[hit])) / mass_tot,
        soc_claim_co_capture=float(np.nansum(bio[hit & soc_tail])) / mass_tot,
        independent_selection=float(np.nansum(bio[hit & ~soc_tail])) / mass_tot,
        missed=float(np.nansum(bio[bio_tail & ~s])) / mass_tot,
    ))
D = pd.DataFrame(rows).set_index("arm")
print(D.round(4).to_string())
print(f"\n(context: {100*float((bio_tail & soc_tail).sum() / bio_tail.sum()):.1f}% of biomass-tail "
      f"CELLS lie inside the m_soc theta-tail; mass columns are fractions of total tail mass)")

MASS_SOC, MASS_BIO = float(np.nansum(soc[pu])), float(np.nansum(bio[pu]))
FRAC_SOC_MASS = MASS_SOC / (MASS_SOC + MASS_BIO)
print(f"regional carbon mass split: SOC {100*FRAC_SOC_MASS:.1f}% / "
      f"biomass {100*(1 - FRAC_SOC_MASS):.1f}%  (spec's ~74/26)")

CAP = float(D.loc["a1_protocol", "capture_mass"])   # the pre-registered rule reads THIS number
if CAP >= 0.90:
    VERDICT, SPLIT_RULE = "TAIL CAPTURED -> (a) mass-proportional", "mass"
elif CAP >= 0.50:
    VERDICT, SPLIT_RULE = "TAIL PARTLY ESCAPING -> (b) equal split", "equal"
else:
    VERDICT, SPLIT_RULE = "TAIL ESCAPING -> STOP: take the numbers back to the chat", "STOP"
print(f"\na1 mass-weighted tail capture = {CAP:.3f}  ->  {VERDICT}")

            tail_cells_selected  capture_cells  capture_mass  soc_claim_co_capture  independent_selection  missed
arm                                                                                                              
a0_control        14,888/14,919         0.9979        0.9981                0.0109                 0.9872  0.0019
a1_protocol       14,917/14,919         0.9999        0.9999                0.0109                 0.9890  0.0001

(context: 1.0% of biomass-tail CELLS lie inside the m_soc theta-tail; mass columns are fractions of total tail mass)
regional carbon mass split: SOC 74.2% / biomass 25.8%  (spec's ~74/26)

a1 mass-weighted tail capture = 1.000  ->  TAIL CAPTURED -> (a) mass-proportional


### CHECKPOINT — confirm the split before §B

`SPLIT` below defaults to the pre-registered rule's outcome. **Override only with a
documented reason** (→ a `methods_log.md` entry in the same session). A `STOP` verdict does
not proceed: the combined total-carbon escalation is a spec amendment decided with the chat.

In [4]:
SPLIT = SPLIT_RULE      # <- Ethan: confirm (or override WITH a methods_log entry)
assert SPLIT in ("mass", "equal"), (
    "diagnostic verdict is STOP -- do not proceed to SS B; take the numbers back to the chat")
frac_soc = FRAC_SOC_MASS if SPLIT == "mass" else 0.5
WITHIN = {"carbon": {"irrecoverable_carbon_m_soc": frac_soc,
                     "irrecoverable_carbon_biomass": 1.0 - frac_soc}}
print(f"within-block split ({SPLIT}): m_soc {100*frac_soc:.1f}% / "
      f"biomass {100*(1 - frac_soc):.1f}% of the carbon block's share")

within-block split (mass): m_soc 74.2% / biomass 25.8% of the carbon block's share


## §B — Block accounting → S0–S4

Four PROACT blocks (`config.BLOCKS`) split the discretionary influence budget; a feature's
intended share = block share × within-block fraction, and since two-regime swing is linear
in w, `w = share / swing_per_unit_w(cap_min, cap_max, t)` (mean-1 normalized). **Outside the
accounting, disclosed:** gHM intactness w=1 (R3-inexpressible) and the EFGs at 1/40 (locked
adequacy foundation) — their *realized* influence is reported in T1, never budgeted.

S1–S3 double one block's share (others scaled down proportionally). S4 doubles carbon **and**
relaxes θ 5×→3× (D2) — the target is an archive lookup, demonstrating spec D2's
budget-independence requirement. Intended-vs-realized influence is verified when each
scenario first *solves* (Gate 2 onward); the biomass weight is iterated once if the miss is
large (Claim C's stated procedure).

In [5]:
# ---- S0: equal discretionary influence per block -------------------------------------------
BASE_SHARES = {b: 1.0 / len(config.BLOCKS) for b in config.BLOCKS}
T_S0 = dict(config.TARGETS)                    # {m_soc: 0.332} -- the frozen protocol target
S0 = lc.scenario_weights(BASE_SHARES, within_block=WITHIN, targets=T_S0)
print(S0.round(4).to_string(index=False))
print("\noutside the accounting (fixed, disclosed): human_modification w=1 (R3-inexpressible); "
      "40 EFGs @ 1/40 (locked adequacy foundation)")

                     feature        block  intended_share     t  per_unit_swing      w  realized_share
   climate_type_macrorefugia core_habitat          0.2500 1.000          0.4224 1.4600          0.2500
  transboundary_connectivity connectivity          0.1250 1.000          0.4607 0.6693          0.1250
           climate_corridors connectivity          0.1250 1.000          0.2632 1.1714          0.1250
  irrecoverable_carbon_m_soc       carbon          0.1855 0.332          0.9847 0.4646          0.1855
irrecoverable_carbon_biomass       carbon          0.0645 1.000          0.8014 0.1986          0.0645
          aoh_richness_birds biodiversity          0.1250 1.000          0.2321 1.3286          0.1250
        aoh_richness_mammals biodiversity          0.1250 1.000          0.1806 1.7075          0.1250

outside the accounting (fixed, disclosed): human_modification w=1 (R3-inexpressible); 40 EFGs @ 1/40 (locked adequacy foundation)


In [6]:
# ---- S4's target: theta 5x -> 3x is an ARCHIVE LOOKUP (spec D2) ----------------------------
# The audit archived budget-independent curves precisely so a different theta costs zero
# recomputation: find where the marginal-density trajectory falls below 3x the regional mean
# and read the captured fraction there.
z = np.load(AUDIT_OBJ / "feature_audit.npz")
area  = z["irrecoverable_carbon_m_soc__area"]
ratio = z["irrecoverable_carbon_m_soc__dens_ratio"]
cap   = z["irrecoverable_carbon_m_soc__captured"]
i3 = int(np.argmax(ratio < 3.0))               # trajectories are density-descending
T_S4_MSOC = float(cap[i3])
print(f"theta=3x: {100*float(area[i3]):.1f}% of region -> m_soc target {T_S4_MSOC:.3f}")
assert abs(T_S4_MSOC - 0.552) < 0.005, "theta=3x lookup drifted from the verified 0.552 -- STOP"

theta=3x: 9.8% of region -> m_soc target 0.552


In [7]:
# ---- the scenario family + T1 skeleton -----------------------------------------------------
def doubled(block):
    """Spec s3.1: the named block's share doubled, the others scaled down proportionally."""
    n = len(config.BLOCKS)
    return {b: (2.0 / n if b == block else (1.0 - 2.0 / n) / (n - 1)) for b in config.BLOCKS}

SCENARIOS = {
    "S0_balanced":     (BASE_SHARES,             dict(T_S0)),
    "S1_core_habitat": (doubled("core_habitat"), dict(T_S0)),
    "S2_connectivity": (doubled("connectivity"), dict(T_S0)),
    "S3_biodiversity": (doubled("biodiversity"), dict(T_S0)),
    "S4_carbon":       (doubled("carbon"),       {"irrecoverable_carbon_m_soc": round(T_S4_MSOC, 3)}),
}
TBL = {name: lc.scenario_weights(sh, within_block=WITHIN, targets=tg)
       for name, (sh, tg) in SCENARIOS.items()}
# (scenario_weights asserts realized == intended shares internally -- the wiring identity)

t1 = pd.concat({k: v.set_index("feature")["w"] for k, v in TBL.items()}, axis=1)
t1.insert(0, "block", TBL["S0_balanced"].set_index("feature")["block"])
print("T1 skeleton -- derived weights (mean-1 per scenario)")
print(t1.round(3).to_string())
print("\ntargets: " + "; ".join(
    f"{k}: {tg if tg else 'none'}" for k, (_, tg) in SCENARIOS.items()))

T1 skeleton -- derived weights (mean-1 per scenario)
                                     block  S0_balanced  S1_core_habitat  S2_connectivity  S3_biodiversity  S4_carbon
feature                                                                                                              
climate_type_macrorefugia     core_habitat        1.460            3.091            0.957            0.782      1.229
transboundary_connectivity    connectivity        0.669            0.472            1.316            0.358      0.563
climate_corridors             connectivity        1.171            0.827            2.303            0.627      0.986
irrecoverable_carbon_m_soc          carbon        0.465            0.328            0.304            0.249      1.166
irrecoverable_carbon_biomass        carbon        0.199            0.140            0.130            0.106      0.501
aoh_richness_birds            biodiversity        1.329            0.937            0.871            2.134      1.118
aoh

In [8]:
# ---- freeze the derivation: spec/scenarios_v1.json (the Gate-3 manifest's input) -----------
payload = {"_meta": dict(
    derived_utc=datetime.now(timezone.utc).isoformat(),
    audit_created_utc=CONSTS["created_utc"],
    n_pu=CONSTS["n_pu"],
    budget_pct=config.BUDGET_PCT,
    split_rule=SPLIT,
    frac_soc=round(frac_soc, 6),
    diagnostic_a1_tail_capture_mass=round(CAP, 6),
    normalization="mean-1 over blocked features; outside fixed (gHM w=1, EFG 1/40)",
    layer_sha256=CONSTS["layer_sha256"],
)}
for name, (sh, tg) in SCENARIOS.items():
    d = TBL[name]
    payload[name] = dict(
        block_shares={k: round(v, 6) for k, v in sh.items()},
        within_block={b: {f: round(x, 6) for f, x in m.items()} for b, m in WITHIN.items()},
        targets=tg,
        weights={r.feature: round(r.w, 6) for r in d.itertuples()},
        intended_shares={r.feature: round(r.intended_share, 6) for r in d.itertuples()},
    )
out = SPEC / "scenarios_v1.json"
out.write_text(json.dumps(payload, indent=2))
print(f"wrote {out.relative_to(ROOT)}  ({len(SCENARIOS)} scenarios; split_rule={SPLIT})")

wrote analyses/y2y/spec/scenarios_v1.json  (5 scenarios; split_rule=mass)


## §C — Climate realizations present + separated

Completeness check for the Gate-1 record (built by 02's closing section): both realization
layers exist, leverage sits inside the measured six-realization span (0.42–0.52), and the
top-30% Jaccard between the two axis levels ≈ 0.574 (D6).

In [9]:
missing = [k for k in config.CLIMATE_REALIZATIONS
           if not (config.REALIZATIONS_DIR / f"macrorefugia_{k}.tif").exists()]
assert not missing, f"realization layers missing: {missing} -- run 02's closing section first"
tops = {}
for k in config.CLIMATE_REALIZATIONS:
    v = lc._read(config.REALIZATIONS_DIR / f"macrorefugia_{k}.tif")[pu]
    cmin, cmax, lev = lc.leverage_of(v)
    tops[k] = v >= np.nanquantile(v, 1.0 - config.BUDGET_PCT)
    print(f"{k}: leverage {lev:.3f} [{cmin:.3f}, {cmax:.3f}]")
a, b = (tops[k] for k in config.CLIMATE_REALIZATIONS)
print(f"top-{100*config.BUDGET_PCT:.0f}% Jaccard = {float((a & b).sum() / (a | b).sum()):.3f} "
      f"(D6 measured 0.574)")

245_2071_2100: leverage 0.482 [0.116, 0.599]
585_2071_2100: leverage 0.422 [0.135, 0.558]
top-30% Jaccard = 0.574 (D6 measured 0.574)


## Next

Gate 1's deliverables are complete when this notebook has run clean: the diagnostic verdict,
the confirmed split, `spec/scenarios_v1.json`, and the realization layers. **Report the §A
numbers + the T1 skeleton back before Gate 2** (the first 1 km pool run: reference formulation S0,
k=50, g=5% — pool-cost measurement, the decisive degeneracy test, and the E4 seed; its fail
branch is live per spec §2.9). Fill the pending R-entries in `spec/results_log.md` with the
measured numbers in the same session.